In [78]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import clear_output
from matplotlib.animation import FuncAnimation


In [74]:
#Parameters:
N_s = 100
sigma = 2*np.pi/N_s
h_b = 0
beta = 200
v0 = 10
dt = 0.3
L = 1000
nu = 1
h0 = 0.05
target = np.pi / 2 

In [ ]:
#Nodes
nodes = np.zeros(N_s)
alpha = np.zeros(N_s)
J = np.zeros((N_s, N_s))
u = np.zeros(N_s)
u_next = np.zeros(N_s)
h = np.zeros(N_s)
v = np.zeros(2)

for i in range(N_s):
    alpha[i] = (2*np.pi *i)/N_s

for i in range(N_s):
    for j in range(N_s):
        J[i,j] = (np.cos(alpha[i] - alpha[j])) - nu


In [ ]:
#update
clear_output()
T = 500

for t in range(T):
    for i in range(N_s):
        angular_diff = alpha[i] - target
        # Wrap to [-pi, pi]
        angular_diff = np.arctan2(np.sin(angular_diff), np.cos(angular_diff))
        h[i] = (h0 / np.sqrt(2 * np.pi * sigma**2)) * np.exp(-angular_diff**2 / (2 *sigma**2))
    for i in range(N_s):    
        excitation_term = 0

        for j in range (N_s):
            excitation_term += J[i][j] * np.tanh(beta * u[j])
        
        u_next[i] = u[i] + dt * (-u[i] + (1/N_s) * excitation_term - h_b + h[i])
    u = u_next.copy()
    
    v = np.zeros(2)
    for i in range(N_s):
        v += max(0, np.tanh(beta * u[i])) * np.array([np.cos(alpha[i]), np.sin(alpha[i])])
    v *= v0/N_s
    
    
# print(u)    
# print(np.argmax(u))
    
plot_ring(u, alpha, beta)





In [71]:
def plot_ring(u, alpha, beta):
    firing_rate = np.maximum(0, np.tanh(beta * u))  # the actual output
    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={'projection': 'polar'})
    ax.plot(np.append(alpha, alpha[0]), np.append(firing_rate, firing_rate[0]), color='blue')
    ax.fill_between(np.append(alpha, alpha[0]), 0, np.append(firing_rate, firing_rate[0]), alpha=0.3)
    ax.set_ylim(0, 1.1)
    ax.set_title('Firing rate around the ring')
    plt.tight_layout()
    plt.savefig('ring.png')
    plt.close()